# 19：保存済みTabDiffモデルを選択して仮想患者データを生成
18で保存したモデルを一覧から選び、再学習せずに仮想データを作ります。`notebooks`フォルダに置き、上から実行してください。

選択には一覧の「番号」を使います。同じ実行の`preprocessing.pkl`・公式前処理済みデータ・チェックポイント設定を必ず組み合わせます。元のParquetファイルの再読込は不要です。
初期設定は10,000例。出力は`artifacts/19_tabdiff_generate/実行時刻/`に保存します。18の学習結果を上書きしません。
修正版：通常モデルとEMAモデルの種別を表示します。同じ学習実行で両方を選び、同じ生成件数・GENERATION_SEEDで比較してください。欠損率を強制的に調整する処理は行いません。


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, subprocess, time, hashlib, re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RUNS_ROOT = PROJECT / "artifacts/18_tabdiff"
TABDIFF_REPO = PROJECT / "external/TabDiff"
TABDIFF_PYTHON = Path(sys.executable)
N_GENERATE = 10_000
SAMPLE_BATCH_SIZE = 512
GPU = 0
assert RUNS_ROOT.is_dir(), f"18の保存先を確認してください: {RUNS_ROOT}"
assert (TABDIFF_REPO / "main.py").is_file(), f"TabDiffの場所を確認してください: {TABDIFF_REPO}"
GENERATION_SEED = 42  # 通常・EMA比較では同じ値に設定

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024**2), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
# 実行設定のdatanameからモデルを探します。古いcheckpoint.txtの絶対パスに依存しません。
records, skipped = [], []
for config_path in sorted(RUNS_ROOT.glob("*/run_config.json")):
    run_dir = config_path.parent
    try:
        cfg = json.loads(config_path.read_text(encoding="utf-8"))
        name, exp = cfg["dataname"], cfg["exp_name"]
        if any(Path(v).name != v for v in [name, exp]):
            raise ValueError("データ名・実験名にパスが含まれています")
        ckpt_dir = TABDIFF_REPO / "tabdiff/ckpt" / name / exp
        if not (run_dir / "preprocessing.pkl").is_file():
            raise FileNotFoundError("preprocessing.pklなし")
        if not (ckpt_dir / "config.pkl").is_file():
            raise FileNotFoundError("モデルのconfig.pklなし")
        data_dir = TABDIFF_REPO / "data" / name
        required = ["info.json", "X_num_train.npy", "X_cat_train.npy", "y_train.npy",
                    "X_num_test.npy", "X_cat_test.npy", "y_test.npy"]
        if any(not (data_dir / f).is_file() for f in required):
            raise FileNotFoundError("公式前処理済みデータが不足")
        paths = sorted(p for p in ckpt_dir.glob("*.pt")
                       if p.name.startswith(("final_ema_", "best_ema_model_", "best_model_", "model_")))
        if not paths:
            raise FileNotFoundError("生成に使用できる重みなし")
        for path in paths:
            match = re.search(r"_(\d+)\.pt$", path.name)
            records.append({"番号": len(records), "学習実行": run_dir.name,
                "モデル": path.name, "種別": "EMA" if "ema" in path.name else "通常",
                "EMA更新": cfg.get("ema_update", "旧版/不明"), "epoch": int(match.group(1)) if match else None,
                "サイズMB": round(path.stat().st_size/1024**2, 1),
                "重みパス": str(path), "設定パス": str(config_path),
                "データ名": name})
    except (KeyError, ValueError, OSError) as exc:
        skipped.append({"学習実行": run_dir.name, "理由": str(exc)})
models = pd.DataFrame(records)
if models.empty:
    display(pd.DataFrame(skipped))
    raise RuntimeError("使用可能なモデルがありません。学習保存先と公式前処理済みデータを確認してください。")
display(models[["番号", "学習実行", "モデル", "種別", "EMA更新", "epoch", "サイズMB"]])
if skipped:
    print("一覧に載せなかった実行:")
    display(pd.DataFrame(skipped))

## 使用するモデルを選択
上の一覧にある「番号」を`MODEL_NUMBER`に入力してください。生成件数やGPUも最初の設定セルで変更できます。
途中で止まった学習は、保存済み重みがある場合だけ一覧に表示されます。`final_ema`は学習終了時のEMA重みです。

In [ ]:
MODEL_NUMBER = 0  # ← 一覧から選んだ番号に変更

if MODEL_NUMBER not in models["番号"].values:
    raise ValueError("一覧にある番号を指定してください。")
selected = models.loc[models["番号"].eq(MODEL_NUMBER)].iloc[0]
CHECKPOINT = Path(selected["重みパス"])
SOURCE_CONFIG = Path(selected["設定パス"])
SOURCE_RUN = SOURCE_CONFIG.parent
source_config = json.loads(SOURCE_CONFIG.read_text(encoding="utf-8"))
preprocessing = pd.read_pickle(SOURCE_RUN / "preprocessing.pkl")
schema = preprocessing["schema"]
constants = preprocessing["constants"]
output_columns = preprocessing["columns"]
# 同じ実行の内部列と公式メタデータが一致することを確認します。
info_path = TABDIFF_REPO / "data" / source_config["dataname"] / "info.json"
info = json.loads(info_path.read_text(encoding="utf-8"))
expected_keys = set()
for spec in schema.values():
    expected_keys.add(spec["key"])
    if "mask" in spec:
        expected_keys.add(spec["mask"])
if expected_keys != set(info["column_names"]):
    raise ValueError("前処理情報とモデル入力の列が一致しません。")
if not isinstance(N_GENERATE, int) or N_GENERATE <= 0:
    raise ValueError("生成件数は正の整数にしてください。")
if SAMPLE_BATCH_SIZE <= 0:
    raise ValueError("生成バッチサイズは正にしてください。")
print("選択モデル:", CHECKPOINT)
print("学習時の前処理情報:", SOURCE_RUN / "preprocessing.pkl")
print("生成件数:", N_GENERATE)
# 新18で保存した資産ハッシュを検証（旧版は記録がないためスキップ）。
assets_path = SOURCE_RUN / "training_assets.json"
training_assets = json.loads(assets_path.read_text()) if assets_path.is_file() else {}
checks = [("preprocessing_sha256", SOURCE_RUN / "preprocessing.pkl"),
          ("train_csv_sha256", TABDIFF_REPO / "data" / source_config["dataname"] / f"{source_config['dataname']}.csv"),
          ("holdout_csv_sha256", TABDIFF_REPO / "data" / source_config["dataname"] / "holdout.csv")]
for key, path in checks:
    if key in training_assets and sha256_file(path) != training_assets[key]:
        raise ValueError(f"学習後にファイルが変更されています: {path}")
print("選択モデル種別:", selected["種別"], "/ EMA更新:", selected["EMA更新"])


In [ ]:
# 毎回独立した出力先を作成し、選択した重みを明示して設定を作ります。
OUT = PROJECT / "artifacts/19_tabdiff_generate" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "checkpoint.txt").write_text(str(CHECKPOINT.resolve()), encoding="utf-8")
config = {**source_config,
    "repo": str(TABDIFF_REPO.resolve()), "gpu": GPU,
    "n_generate": N_GENERATE, "sample_batch": SAMPLE_BATCH_SIZE,
    "checkpoint_manifest": str(OUT / "checkpoint.txt"),
    "generated_csv": str(OUT / "generated_encoded.csv"), "seed": GENERATION_SEED}
CONFIG = OUT / "run_config.json"
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")
# モデルの識別情報を保存します。
digest = hashlib.sha256()
with CHECKPOINT.open("rb") as f:
    for block in iter(lambda: f.read(1024**2), b""):
        digest.update(block)
provenance = {"source_run": str(SOURCE_RUN), "checkpoint": str(CHECKPOINT),
              "checkpoint_sha256": digest.hexdigest(),
              "preprocessing": str(SOURCE_RUN / "preprocessing.pkl"),
              "n_generate": N_GENERATE, "seed": GENERATION_SEED,
              "model_type": selected["種別"], "ema_update": source_config.get("ema_update", "旧版/不明"),
              "training_assets": training_assets,
              "note": "前処理・重みを再利用。学習は実行しない。乱数固定のため同じ条件の再生成は同じ結果になり得る。"}
(OUT / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
runner_source = "\nimport os, sys, json, argparse, importlib\nfrom pathlib import Path\nimport torch\nimport random\nimport numpy as np\ncfg = json.loads(Path(sys.argv[1]).read_text())\nrandom.seed(cfg['seed'])\nnp.random.seed(cfg['seed'])\ntorch.manual_seed(cfg['seed'])\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(cfg['seed'])\nsys.path.insert(0, cfg['repo'])\nos.chdir(cfg['repo'])\n# バッチごとの大量出力を抑止\nimport tqdm\n_BaseTqdm = tqdm.tqdm\nclass QuietTqdm(_BaseTqdm):\n    def __init__(self, *args, **kwargs):\n        kwargs['disable'] = True\n        super().__init__(*args, **kwargs)\ntqdm.tqdm = QuietTqdm\nmodule = importlib.import_module('tabdiff.main')\n# PyTorchの版による引数の違いを吸収します（公式ファイルは変更しません）。\n# verbose非対応の版でのみ引数を除去し、他の引数はそのまま渡します。\nimport inspect\ntrainer_module = importlib.import_module('tabdiff.trainer')\n_original_plateau = trainer_module.ReduceLROnPlateau\nif 'verbose' not in inspect.signature(_original_plateau).parameters:\n    def compatible_plateau(*args, **kwargs):\n        kwargs.pop('verbose', None)\n        return _original_plateau(*args, **kwargs)\n    trainer_module.ReduceLROnPlateau = compatible_plateau\n    print('PyTorch compatibility: ReduceLROnPlateau verbose argument omitted')\n\nTrainer = module.Trainer\n# test時は学習時のconfig.pklが読まれるため、生成バッチサイズは初期化時に上書き。\noriginal_init = Trainer.__init__\ndef sampling_init(self, *args, **kwargs):\n    kwargs['sample_batch_size'] = cfg['sample_batch']\n    original_init(self, *args, **kwargs)\nTrainer.__init__ = sampling_init\noriginal_load = module.src.load_config\ndef generate_only(self):\n    self.diffusion.eval()\n    with torch.no_grad():\n        frame = self.sample_synthetic(cfg['n_generate'], keep_nan_samples=False)\n    if len(frame) != cfg['n_generate']:\n        raise RuntimeError('Unexpected generated row count')\n    frame.to_csv(cfg['generated_csv'], index=False)\nTrainer.test = generate_only\nmode = sys.argv[2]\nif mode != 'test':\n    raise RuntimeError('Notebook 19 supports generation only')\nif cfg['gpu'] >= 0 and not torch.cuda.is_available():\n    raise RuntimeError('CUDA unavailable. Set GPU=-1 for CPU, or fix CUDA environment.')\nargs = argparse.Namespace(dataname=cfg['dataname'], mode=mode, method='tabdiff',\n    gpu=cfg['gpu'], device=('cuda:%d'%cfg['gpu']) if cfg['gpu']>=0 else 'cpu',\n    debug=False, no_wandb=True, exp_name=cfg['exp_name'], deterministic=False,\n    y_only=False, non_learnable_schedule=False, num_samples_to_generate=cfg['n_generate'],\n    ckpt_path=None, report=False, num_runs=1, impute=False)\nif mode == 'test':\n    args.ckpt_path = Path(cfg['checkpoint_manifest']).read_text().strip()\nmodule.main(args)\n"
RUNNER = OUT / "run_tabdiff.py"
RUNNER.write_text(runner_source, encoding="utf-8")

In [ ]:
# testだけを実行します。進捗バーを抑え、エラーの詳細はログ末尾に表示します。
from collections import deque
log_path = OUT / "generate.log"
env = os.environ.copy()
env.update(WANDB_MODE="disabled", PYTHONUNBUFFERED="1", PYTHONHASHSEED=str(GENERATION_SEED))
print("生成開始:", N_GENERATE, "例 / ログ:", log_path)
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen([str(TABDIFF_PYTHON), str(RUNNER), str(CONFIG), "test"],
        cwd=TABDIFF_REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
    started, last_report = time.monotonic(), time.monotonic()
    try:
        while proc.poll() is None:
            time.sleep(2)
            if time.monotonic() - last_report >= 60:
                print(f"生成処理中：経過 {(time.monotonic()-started)/60:.1f} 分", flush=True)
                last_report = time.monotonic()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("生成を中断しました。学習済みモデルは残っています。")
        raise
if proc.returncode:
    with log_path.open(encoding="utf-8", errors="replace") as f:
        print("".join(deque(f, maxlen=80)))
    raise RuntimeError(f"生成失敗：上のエラーとログを確認してください: {log_path}")
print("モデルからの生成完了")

In [ ]:
# 内部コードを日本語列名・カテゴリー・日時・欠損へ復元します。
generated = pd.read_csv(OUT / "generated_encoded.csv", keep_default_na=False)
if len(generated) != N_GENERATE:
    raise ValueError(f"生成件数が不一致: {len(generated)}")
df_synthetic = pd.DataFrame(index=range(N_GENERATE))
for column, spec in schema.items():
    values = generated[spec["key"]]
    if spec["kind"] == "category":
        values = values.astype(str)
        unknown = ~values.isin(list(spec["decode"]) + ["missing_category"])
        if unknown.any():
            raise ValueError(f"未知の生成カテゴリー: {column}")
        decoded = values.map(spec["decode"])
        dtype = spec["dtype"]
        if isinstance(dtype, pd.CategoricalDtype):
            # 入力カテゴリ値が数値の場合も元の型へ戻す。
            reverse = {str(v): v for v in dtype.categories}
            decoded = decoded.map(reverse)
            df_synthetic[column] = pd.Categorical(decoded, dtype=dtype)
        else:
            df_synthetic[column] = decoded.astype("string")
    else:
        values = pd.to_numeric(values, errors="raise").astype(float)
        if not np.isfinite(values).all():
            raise ValueError(f"非有限の生成値: {column}")
        values = values.clip(spec["minimum"], spec["maximum"])
        if spec["integer"]:
            values = values.round()
        if "mask" in spec:
            values = values.mask(generated[spec["mask"]].eq("missing_yes"))
        if spec["kind"] == "datetime":
            df_synthetic[column] = pd.Timestamp("1970-01-01") + pd.to_timedelta(values, unit="D")
        elif spec["integer"]:
            df_synthetic[column] = values.astype("Int64")
        else:
            df_synthetic[column] = values
for column, value in constants.items():
    df_synthetic[column] = value
    # 新18で保存した定数列の型を復元。旧版は値のみ復元します。
    dtype = preprocessing.get("dtypes", {}).get(column)
    if isinstance(dtype, pd.CategoricalDtype):
        df_synthetic[column] = pd.Categorical(df_synthetic[column], dtype=dtype)
df_synthetic = df_synthetic[output_columns]
# 復元時点の表を保持し、後処理で増えた欠損と区別します。
restored_before_postprocessing = df_synthetic.copy()
restored_before_postprocessing.to_pickle(OUT / "generated_restored_before_postprocessing.pkl")
# 日付と年は独立生成の不一致を残さず、生成した発生日を基準に再構成します。
if "発生年月日ウツタイン" in df_synthetic:
    event = pd.to_datetime(df_synthetic["発生年月日ウツタイン"], errors="coerce").dt.normalize()
    df_synthetic["発生年月日ウツタイン"] = event
    if "年" in df_synthetic:
        df_synthetic["年"] = event.dt.year.astype("Int64")
    if "覚知ウツタイン" in df_synthetic:
        call = pd.to_datetime(df_synthetic["覚知ウツタイン"], errors="coerce")
        df_synthetic["覚知ウツタイン"] = event + (call - call.dt.normalize()).dt.floor("min")
# 01の時間採用基準を生成結果にも適用します。
for column in df_synthetic:
    if column in ["覚知現着分", "現着接触分", "接触病着分"]:
        df_synthetic[column] = df_synthetic[column].where(df_synthetic[column].between(0,120))
    elif column.startswith("覚知") and column.endswith("分"):
        low, high = (-120,0) if column == "覚知目撃分" else (-120,120)
        df_synthetic[column] = df_synthetic[column].where(df_synthetic[column].between(low,high))
assert len(df_synthetic) == N_GENERATE
# 生成直後の内部表はgenerated_encoded.csvとして別に保持しています。
df_synthetic.to_pickle(OUT / f"utstein_synthetic_{N_GENERATE}.pkl")
df_synthetic.to_csv(OUT / f"utstein_synthetic_{N_GENERATE}.csv", index=False, encoding="utf-8-sig")
print("保存先:", OUT)
display(df_synthetic.head(100).T)
# 生成直後→復元→後処理の欠損率を保存します。
stages = []
for column, spec in schema.items():
    if spec["kind"] == "category":
        raw_missing = generated[spec["key"]].astype(str).eq("missing_category")
    elif "mask" in spec:
        flag = generated[spec["mask"]].astype(str)
        if not flag.isin(["missing_yes", "missing_no"]).all():
            raise ValueError(f"不正な生成欠損フラグ: {column}")
        raw_missing = flag.eq("missing_yes")
    else:
        raw_missing = pd.Series(False, index=generated.index)
    before = restored_before_postprocessing[column].isna()
    after = df_synthetic[column].isna()
    stages.append({"列名": column, "生成フラグ欠損率": float(raw_missing.mean()),
        "復元直後欠損率": float(before.mean()), "最終欠損率": float(after.mean()),
        "フラグと復元不一致件数": int(raw_missing.ne(before).sum()),
        "後処理で増えた欠損件数": int((~before & after).sum())})
missingness_stages = pd.DataFrame(stages)
missingness_stages.to_csv(OUT / "missingness_stages.csv", index=False, encoding="utf-8-sig")
# 20が選択PKLと生成内部表の由来を確認できるよう、出力ハッシュも記録。
provenance.update({"synthetic_sha256": sha256_file(OUT / f"utstein_synthetic_{N_GENERATE}.pkl"),
                   "generated_encoded_sha256": sha256_file(OUT / "generated_encoded.csv"),
                   "restored_before_postprocessing_sha256": sha256_file(OUT / "generated_restored_before_postprocessing.pkl")})
(OUT / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
display(missingness_stages)


In [ ]:
# 元データを再読込せず、生成データの欠損・不整合件数を確認します。
display(pd.DataFrame({"型": df_synthetic.dtypes.astype(str),
                      "欠損数": df_synthetic.isna().sum(),
                      "欠損率": df_synthetic.isna().mean()}))
issues = {}
if {"除細動", "施行回数"}.issubset(df_synthetic):
    issues["除細動なし・施行回数1以上"] = int((df_synthetic["除細動"].eq("なし") & pd.to_numeric(df_synthetic["施行回数"], errors="coerce").ge(1)).sum())
if {"バイスタンダーCPR", "心臓マッサージ"}.issubset(df_synthetic):
    issues["バイスタンダーCPRなし・心臓マッサージあり"] = int((df_synthetic["バイスタンダーCPR"].eq("なし") & df_synthetic["心臓マッサージ"].eq("チェックあり")).sum())
report = pd.Series(issues, dtype="int64", name="件数").to_frame()
display(report)
report.to_csv(OUT / "synthetic_inconsistencies.csv", encoding="utf-8-sig")
print("保存先:", OUT)

## 保存されるもの
- `utstein_synthetic_10000.pkl` / `.csv`：型・欠損の復元および日時・経過分数の後処理を行った仮想患者データ。件数を変えるとファイル名も変わります。
- `generated_encoded.csv`：モデルが生成した内部形式の表。後処理前の値です。
- `provenance.json`：使用モデル・前処理情報の保存先・重みのSHA256。
- `generate.log`：生成中の詳細ログ。

カテゴリ・欠損復元、整数丸め、学習範囲へのクリップ、発生日からの年・覚知日再構成は18と同様です。定数カテゴリのdtypeは旧18の保存情報だけでは復元できないため、値を復元します。
同じモデルを使っても論理的不一致がゼロとは限りません。末尾の件数で確認してください。

このノートブックは実データ・GPU生成をこちらでは実行していません。構文・一覧選択・設定連携・模擬生成表の復元を確認しています。